# BioLogic EC-Lab Experimental Electrochemistry Workflow

This interactive research notebook provides an end-to-end scientific processing pipeline for experimental electrochemistry data acquired from **BioLogic** potentiostats using **EC-Lab** software.

### Analysis Modules Covered:
1. **Hydrogen Evolution Reaction (HER) LSV**: Automatic `.mpt` ASCII parsing, RHE conversion, and cathodic overpotential extraction ($\eta_{-10}, \eta_{-50}$).
2. **Potentio Electrochemical Impedance Spectroscopy (PEIS)**: Dual-panel stacked Bode plots (magnitude $|Z|$ and phase angle $\phi$ vs frequency).
3. **Lin-KK Kramers-Kronig Consistency Validation**: Rigorous statistical verification of stationarity, linearity, and causality using Schönleber's criterion.
4. **Galvanostatic Chronopotentiometry (CP)**: Durability monitoring at $-10\,\mathrm{mA/cm^2}$ and potential drift rate estimation ($\mathrm{mV/h}$).


In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt

from scripts.utils import apply_publication_style, create_publication_figure
from parsers.electrochemistry import load_electrochemistry_data
from modules.electrochemistry import (
    plot_lsv,
    extract_overpotential_at_current,
    plot_bode,
    run_kramers_kronig_test,
    plot_chronopotentiometry,
    calculate_cp_stability_metrics,
)

apply_publication_style()
data_dir = project_root / "data" / "examples" / "electrochemistry"
print("BioLogic processing pipeline loaded.")


## 1. Hydrogen Evolution Reaction (HER) Polarization Curves

Load raw BioLogic `.mpt` export (`biologic_her_lsv.mpt`) and extract cathodic overpotentials ($\eta_{-10}$ and $\eta_{-50}$).


In [ ]:
her_file = data_dir / "biologic" / "biologic_her_lsv.mpt"
data_her = load_electrochemistry_data(her_file, instrument="biologic", technique="LSV")

print(f"Software: {data_her.instrument_software}")
print(f"Electrode Area: {data_her.electrode_area_cm2} cm^2")
print(f"Reference: {data_her.potential_reference_raw} | pH: {data_her.ph}")

pot_her_rhe = data_her.get_potential_vs_rhe()
j_her = data_her.get_current_density(unit="mA/cm^2")

eta_10_her = extract_overpotential_at_current(pot_her_rhe, j_her, target_j=10.0, reaction="HER")
eta_20_her = extract_overpotential_at_current(pot_her_rhe, j_her, target_j=20.0, reaction="HER")

print(f"\n[Metrics] Overpotential eta_-10: {eta_10_her:.1f} mV")
print(f"[Metrics] Overpotential eta_-20: {eta_20_her:.1f} mV" if eta_20_her is not None else "[Metrics] Overpotential eta_-20: N/A")

ax, bm = plot_lsv(
    data_her,
    reaction="HER",
    ir_compensate=False,
    target_currents=(10.0, 20.0),
    title="BioLogic EC-Lab - HER Polarization (0.5 M H2SO4)",
)
plt.show()


## 2. PEIS Stacked Bode Analysis & Lin-KK Validation

Load BioLogic `.mpt` impedance spectrum (`biologic_peis.mpt`), visualize Bode frequency responses, and validate system stability via Lin-KK Kramers-Kronig transform.


In [ ]:
peis_file = data_dir / "biologic" / "biologic_peis.mpt"
data_peis = load_electrochemistry_data(peis_file, instrument="biologic", technique="EIS")

f = data_peis.frequency_Hz
zr = data_peis.impedance_real_ohm
zi = data_peis.impedance_imag_ohm

kk_res = run_kramers_kronig_test(f, zr, zi)

print(f"Lin-KK Valid: {kk_res['valid']}")
print(f"Lin-KK mu score: {kk_res['mu']:.4f}")
print(f"Number of RC Elements: {kk_res['num_elements']}")

fig, (ax_mag, ax_phase) = plot_bode(
    data_peis,
    title="BioLogic EC-Lab - PEIS Bode Representation",
)
plt.show()


## 3. Galvanostatic Chronopotentiometry (CP) Stability

Measure potential stability during continuous HER operation at $j = -10\,\mathrm{mA/cm^2}$ over 1 hour. Extract potential drift rate in $\mathrm{mV/h}$.


In [ ]:
cp_file = data_dir / "biologic" / "biologic_stability_cp.mpt"
data_cp = load_electrochemistry_data(cp_file, instrument="biologic", technique="CP")

t = data_cp.time_s
v = data_cp.get_potential_vs_rhe(reference="ag/agcl", ph=0.3)

metrics_cp = calculate_cp_stability_metrics(t, v)

print(f"Potential Drift: {metrics_cp['potential_drift_mV']:.2f} mV")
print(f"Drift Rate: {metrics_cp['drift_rate_mV_per_h']:.2f} mV/h")

fig, ax = plot_chronopotentiometry(
    data_cp,
    ph=0.3,
    current_label="j = -10 mA/cm^2",
    show_drift_rate=True,
    title="BioLogic EC-Lab - 1-Hour HER Galvanostatic Durability",
)
plt.show()
